<a href="https://colab.research.google.com/github/shaestasaleem/flyrank-machine-learning-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaestasaleem/flyrank-machine-learning-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [20]:
# ML-04 setup: connect DuckDB to the gated Hugging Face warehouse

!pip -q install --upgrade duckdb

import duckdb
from google.colab import userdata
from IPython.display import display

# Read token safely from Colab Secrets
hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN not found. Add it in the Colab Secrets panel "
        "and enable notebook access."
    )

# Create DuckDB connection
con = duckdb.connect()

# Enable access to remote Parquet files
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# Safely register the Hugging Face token with DuckDB
safe_token = hf_token.replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_token (
        TYPE HUGGINGFACE,
        TOKEN '{safe_token}'
    );
""")

# Warehouse paths
WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

MARCH_DAILY = (
    f"{WAREHOUSE}/fact_content_daily_performance/"
    "month=2026-03/data_0.parquet"
)

DIM_CONTENT = f"{WAREHOUSE}/dim_content.parquet"
DIM_CLIENTS = f"{WAREHOUSE}/dim_clients.parquet"

print("DuckDB connection created successfully.")
print("Development month: March 2026")
print("No raw rows or private identifiers are being displayed.")

DuckDB connection created successfully.
Development month: March 2026
No raw rows or private identifiers are being displayed.


In [21]:
# Inspect table schemas without displaying client-level data

daily_schema = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{MARCH_DAILY}')
""").df()

content_schema = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{DIM_CONTENT}')
""").df()

client_schema = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{DIM_CLIENTS}')
""").df()

print("fact_content_daily_performance — March 2026 columns")
display(daily_schema[["column_name", "column_type", "null"]])

print("\ndim_content columns")
display(content_schema[["column_name", "column_type", "null"]])

print("\ndim_clients columns")
display(client_schema[["column_name", "column_type", "null"]])

fact_content_daily_performance — March 2026 columns


,column_name,column_type,null
0,report_date,DATE,YES
1,client_hash_id,VARCHAR,YES
2,content_hash_id,VARCHAR,YES
3,client_has_gsc,BOOLEAN,YES
4,client_has_ga4,BOOLEAN,YES
5,gsc_data_available,BOOLEAN,YES
6,ga4_data_available,BOOLEAN,YES
7,gsc_impressions,BIGINT,YES
8,gsc_clicks,BIGINT,YES
9,gsc_sum_position,BIGINT,YES



dim_content columns


,column_name,column_type,null
0,client_hash_id,VARCHAR,YES
1,content_hash_id,VARCHAR,YES
2,keyword_hash_id,VARCHAR,YES
3,url_hash_id,VARCHAR,YES
4,keyword_char_count,BIGINT,YES
5,keyword_token_count,BIGINT,YES
6,url_char_count,BIGINT,YES
7,content_created_date,DATE,YES
8,content_updated_date,DATE,YES
9,content_type,VARCHAR,YES



dim_clients columns


,column_name,column_type,null
0,client_hash_id,VARCHAR,YES
1,is_active,BOOLEAN,YES
2,has_gsc_access,BOOLEAN,YES
3,has_ga4_access,BOOLEAN,YES
4,access_profile,VARCHAR,YES
5,client_created_date,DATE,YES
6,client_updated_date,DATE,YES
7,gsc_data_start,DATE,YES
8,ga4_data_start,DATE,YES


## 1. Unit of analysis + time window

### Data contract

**Lane:** Refresh / Content Opportunity Scoring

**Source unit of analysis:** In `fact_content_daily_performance`, one row represents one report date for one pseudonymized client and one pseudonymized content item.

**Feature-frame unit:** After aggregation, one row will represent one pseudonymized client–content item summarized over the March 2026 development window.

**Tables used:**  
- `fact_content_daily_performance` for daily search and engagement performance  
- `dim_content` for content-level context  
- `dim_clients` only for checking when GSC and GA4 history became available

**Development window:** March 1–31, 2026.

**What I want to rank:** Content items by review priority, so a content editor can decide which items should be investigated or refreshed first.

**Deliberately excluded:** June 2026 data, because it is the sealed final test month and must not influence feature or label development.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Verification Query 1 of 3: confirm the daily table grain

grain_duplicates = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS rows_at_claimed_grain
    FROM read_parquet('{MARCH_DAILY}')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

display(grain_duplicates)

if grain_duplicates.empty:
    print(
        "PASS: No duplicate rows were found at the claimed grain "
        "(report_date × client_hash_id × content_hash_id)."
    )
else:
    print(
        "REVIEW: Duplicate grain keys were found. "
        "The unit of analysis must be reconsidered."
    )


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,rows_at_claimed_grain


PASS: No duplicate rows were found at the claimed grain (report_date × client_hash_id × content_hash_id).


## 2. Fields: feature / label / context / excluded

### Field roles and decision moment

**Decision moment:** End of March 24, 2026.

**Historical feature window:** March 1–24, 2026.

**Recent comparison window:** March 18–24, 2026.

**Future outcome window:** March 25–31, 2026.

#### Features — maximum five

1. **historical_impressions**  
   Total GSC impressions observed from March 1–24.

2. **historical_clicks**  
   Total GSC clicks observed from March 1–24.

3. **historical_ctr**  
   Historical clicks divided by historical impressions.

4. **historical_avg_position**  
   Impression-weighted average Google Search position during the historical window.

5.  **pre_decision_click_change_7d**  
Difference between clicks observed during March 18–24 and March 11–17. It is knowable because both periods end before the decision moment.

#### Label / proxy

**is_future_click_decline**

This equals 1 when clicks during March 25–31 are lower than clicks during March 18–24; otherwise, it equals 0.

This is a directional proxy for identifying content that may need review. It does not prove that the content itself caused the decline or that every declining item should be refreshed.

#### Context fields

- `client_hash_id`: keeps records separated by pseudonymized client
- `content_hash_id`: identifies the pseudonymized content item
- `report_date`: defines the historical and future windows
- `gsc_data_available`: confirms that GSC data was available

#### Deliberately excluded

- Clicks, impressions, CTR, or position from March 25–31 are excluded from the feature set because they occur after the decision moment.
- The label and any column derived from the label are excluded from honest model features.
- June 2026 is excluded because it is the sealed final test month.
- Client and content hashes are context keys, not predictive features.
- Raw URLs, client names, domains, and private search queries are not displayed or used.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Define the decision windows and document every planned field

import pandas as pd

FEATURE_START = "2026-03-01"
FEATURE_END = "2026-03-24"

RECENT_START = "2026-03-18"
RECENT_END = "2026-03-24"

LABEL_START = "2026-03-25"
LABEL_END = "2026-03-31"

DECISION_DATE = "2026-03-24"

field_contract = pd.DataFrame(
    [
        {
            "field": "historical_impressions",
            "bucket": "feature",
            "source": "gsc_impressions",
            "available_when": (
                "Knowable at the decision moment because it uses only "
                "March 1–24 observations."
            ),
        },
        {
            "field": "historical_clicks",
            "bucket": "feature",
            "source": "gsc_clicks",
            "available_when": (
                "Knowable at the decision moment because it uses only "
                "March 1–24 observations."
            ),
        },
        {
            "field": "historical_ctr",
            "bucket": "feature",
            "source": "gsc_clicks / gsc_impressions",
            "available_when": (
                "Knowable because both clicks and impressions were observed "
                "before the decision moment."
            ),
        },
        {
            "field": "historical_avg_position",
            "bucket": "feature",
            "source": "gsc_sum_position / gsc_impressions",
            "available_when": (
                "Knowable because it is calculated only from historical "
                "GSC records."
            ),
        },
        {
    "field": "pre_decision_click_change_7d",
    "bucket": "feature",
    "source": "March 18–24 clicks minus March 11–17 clicks",
    "available_when": (
        "Knowable because both seven-day periods occurred "
        "before the March 24 decision moment."
    ),
},
        {
            "field": "is_future_click_decline",
            "bucket": "label / proxy",
            "source": "future 7-day clicks compared with recent 7-day clicks",
            "available_when": (
                "Not available at the decision moment; used only as the "
                "future outcome label."
            ),
        },
        {
            "field": "client_hash_id",
            "bucket": "context",
            "source": "fact table",
            "available_when": "Used only as a pseudonymized grouping key.",
        },
        {
            "field": "content_hash_id",
            "bucket": "context",
            "source": "fact table",
            "available_when": "Used only as a pseudonymized grouping key.",
        },
        {
            "field": "future_clicks_7d",
            "bucket": "excluded",
            "source": "March 25–31 clicks",
            "available_when": (
                "Excluded from features because it occurs after the "
                "decision moment and directly helps create the label."
            ),
        },
        {
            "field": "June 2026 data",
            "bucket": "excluded",
            "source": "sealed final month",
            "available_when": (
                "Excluded from development so the final month remains sealed."
            ),
        },
    ]
)

display(field_contract)

print("Decision date:", DECISION_DATE)
print("Feature window:", FEATURE_START, "to", FEATURE_END)
print("Outcome window:", LABEL_START, "to", LABEL_END)
print("Number of planned model features:", 5)

assert (
    field_contract["bucket"].eq("feature").sum() == 5
), "The assignment allows a maximum of five features."

,field,bucket,source,available_when
0,historical_impressions,feature,gsc_impressions,Knowable at the decision moment because it use...
1,historical_clicks,feature,gsc_clicks,Knowable at the decision moment because it use...
2,historical_ctr,feature,gsc_clicks / gsc_impressions,Knowable because both clicks and impressions w...
3,historical_avg_position,feature,gsc_sum_position / gsc_impressions,Knowable because it is calculated only from hi...
4,pre_decision_click_change_7d,feature,March 18–24 clicks minus March 11–17 clicks,Knowable because both seven-day periods occurr...
5,is_future_click_decline,label / proxy,future 7-day clicks compared with recent 7-day...,Not available at the decision moment; used onl...
6,client_hash_id,context,fact table,Used only as a pseudonymized grouping key.
7,content_hash_id,context,fact table,Used only as a pseudonymized grouping key.
8,future_clicks_7d,excluded,March 25–31 clicks,Excluded from features because it occurs after...
9,June 2026 data,excluded,sealed final month,Excluded from development so the final month r...


Decision date: 2026-03-24
Feature window: 2026-03-01 to 2026-03-24
Outcome window: 2026-03-25 to 2026-03-31
Number of planned model features: 5


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [24]:
# This cell is for CODE (numbers, a query, a check).

# Verification Query 2 of 3:
# Check the March 2026 row count and date span

march_counts = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS distinct_clients,
        COUNT(DISTINCT content_hash_id) AS distinct_content_items,
        COUNT(DISTINCT report_date) AS distinct_report_dates,
        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date
    FROM read_parquet('{MARCH_DAILY}')
""").df()

display(march_counts)

first_date = str(march_counts.loc[0, "first_report_date"])
last_date = str(march_counts.loc[0, "last_report_date"])
days = int(march_counts.loc[0, "distinct_report_dates"])

print(f"Observed date span: {first_date} to {last_date}")
print(f"Distinct report dates observed: {days}")
print("These are measured warehouse counts for the March 2026 slice.")

,total_rows,distinct_clients,distinct_content_items,distinct_report_dates,first_report_date,last_report_date
0,9841378,55,331437,31,2026-03-01,2026-03-31


Observed date span: 2026-03-01 00:00:00 to 2026-03-31 00:00:00
Distinct report dates observed: 31
These are measured warehouse counts for the March 2026 slice.


In [25]:
# Verification Query 3 of 3:
# Check how many March rows survive the required GSC availability filter

availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS rows_after_gsc_is_true,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND gsc_impressions IS NOT NULL
              AND gsc_clicks IS NOT NULL
              AND gsc_sum_position IS NOT NULL
        ) AS rows_with_required_gsc_fields,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE gsc_data_available IS TRUE
            ) / COUNT(*),
            2
        ) AS percent_surviving_availability
    FROM read_parquet('{MARCH_DAILY}')
""").df()

display(availability_check)

total_rows = int(availability_check.loc[0, "total_rows"])
available_rows = int(
    availability_check.loc[0, "rows_after_gsc_is_true"]
)
usable_rows = int(
    availability_check.loc[0, "rows_with_required_gsc_fields"]
)

print(
    f"{available_rows:,} of {total_rows:,} rows survive "
    "the gsc_data_available IS TRUE filter."
)

print(
    f"{usable_rows:,} rows also contain the required "
    "GSC impressions, clicks, and position fields."
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_after_gsc_is_true,rows_with_required_gsc_fields,percent_surviving_availability
0,9841378,3611061,3611061,36.69


3,611,061 of 9,841,378 rows survive the gsc_data_available IS TRUE filter.
3,611,061 rows also contain the required GSC impressions, clicks, and position fields.


In [26]:
# Build the five-feature frame.
# This is feature construction, not an additional verification query.

FEATURE_COLUMNS = [
    "historical_impressions",
    "historical_clicks",
    "historical_ctr",
    "historical_avg_position",
    "pre_decision_click_change_7d",
]

LABEL_COLUMN = "is_future_click_decline"

feature_frame = con.sql(f"""
    WITH available_daily AS (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            gsc_impressions,
            gsc_clicks,
            gsc_sum_position
        FROM read_parquet('{MARCH_DAILY}')
        WHERE gsc_data_available IS TRUE
    ),

    content_windows AS (
        SELECT
            client_hash_id,
            content_hash_id,

            COUNT(DISTINCT report_date) AS available_days_in_march,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-24'
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            ) AS historical_impressions,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-24'
                    THEN COALESCE(gsc_clicks, 0)
                    ELSE 0
                END
            ) AS historical_clicks,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-24'
                    THEN COALESCE(gsc_sum_position, 0)
                    ELSE 0
                END
            ) AS historical_sum_position,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-11'
                                         AND DATE '2026-03-17'
                    THEN COALESCE(gsc_clicks, 0)
                    ELSE 0
                END
            ) AS previous_clicks_7d,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-18'
                                         AND DATE '2026-03-24'
                    THEN COALESCE(gsc_clicks, 0)
                    ELSE 0
                END
            ) AS recent_clicks_7d,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-25'
                                         AND DATE '2026-03-31'
                    THEN COALESCE(gsc_clicks, 0)
                    ELSE 0
                END
            ) AS future_clicks_7d

        FROM available_daily
        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        client_hash_id,
        content_hash_id,

        historical_impressions,
        historical_clicks,

        historical_clicks * 1.0
            / NULLIF(historical_impressions, 0)
            AS historical_ctr,

        historical_sum_position * 1.0
            / NULLIF(historical_impressions, 0)
            AS historical_avg_position,

        recent_clicks_7d - previous_clicks_7d
            AS pre_decision_click_change_7d,

        CASE
            WHEN future_clicks_7d < recent_clicks_7d THEN 1
            ELSE 0
        END AS is_future_click_decline

    FROM content_windows

    WHERE available_days_in_march = 31
      AND historical_impressions > 0
      AND recent_clicks_7d > 0
      AND historical_sum_position IS NOT NULL
""").df()

print(f"Feature-frame rows: {len(feature_frame):,}")
print(f"Model features: {len(FEATURE_COLUMNS)}")

# Do not display pseudonymized identifiers in public output.
display(
    feature_frame[
        FEATURE_COLUMNS + [LABEL_COLUMN]
    ].head()
)

print("\nLabel distribution:")
display(
    feature_frame[LABEL_COLUMN]
    .value_counts()
    .rename_axis("is_future_click_decline")
    .reset_index(name="rows")
)

print("\nLabel percentages:")
display(
    (
        feature_frame[LABEL_COLUMN]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )
    .rename_axis("is_future_click_decline")
    .reset_index(name="percent")
)

assert len(FEATURE_COLUMNS) == 5
assert feature_frame[FEATURE_COLUMNS].isna().sum().sum() == 0

print("\nPASS: The frame contains exactly five non-missing model features.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 27,198
Model features: 5


,historical_impressions,historical_clicks,historical_ctr,historical_avg_position,pre_decision_click_change_7d,is_future_click_decline
0,7542.0,15.0,0.001989,8.516972,1.0,0
1,1463.0,1.0,0.000684,2.627478,1.0,1
2,1138.0,3.0,0.002636,5.109842,0.0,1
3,1129.0,1.0,0.000886,5.656333,1.0,1
4,210.0,2.0,0.009524,8.580952,1.0,1



Label distribution:


,is_future_click_decline,rows
0,1,14973
1,0,12225



Label percentages:


,is_future_click_decline,percent
0,1,55.05
1,0,44.95



PASS: The frame contains exactly five non-missing model features.


### Deliberate leakage experiment

I first trained a simple logistic-regression baseline using only the five features available at the March 24 decision moment.

I then deliberately added `label_copy_leak`, a column copied directly from the future outcome label. This should make the test score jump toward perfect because the model is being given the answer.

The leaked feature is invalid because it would not be available when a content editor makes the decision. I therefore delete it and keep only the honest five-feature score.

In [27]:
# Honest baseline and deliberate leakage experiment
# This is not an additional verification query.

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Keep identifiers out of the model
X_honest = feature_frame[FEATURE_COLUMNS].copy()
y = feature_frame[LABEL_COLUMN].astype(int).copy()

# Split row indices once so both experiments use exactly the same rows
all_indices = np.arange(len(feature_frame))

train_indices, test_indices = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=42,
    stratify=y
)

def make_model():
    return Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    random_state=42
                )
            )
        ]
    )

# ---------------------------------------------------------
# 1. Honest experiment
# ---------------------------------------------------------

honest_model = make_model()

honest_model.fit(
    X_honest.iloc[train_indices],
    y.iloc[train_indices]
)

honest_probabilities = honest_model.predict_proba(
    X_honest.iloc[test_indices]
)[:, 1]

honest_auc = roc_auc_score(
    y.iloc[test_indices],
    honest_probabilities
)

# ---------------------------------------------------------
# 2. Deliberate leakage experiment
# ---------------------------------------------------------

X_leaky = X_honest.copy()

# This column directly copies the answer.
# It would not exist at the real decision moment.
X_leaky["label_copy_leak"] = y

leaky_model = make_model()

leaky_model.fit(
    X_leaky.iloc[train_indices],
    y.iloc[train_indices]
)

leaky_probabilities = leaky_model.predict_proba(
    X_leaky.iloc[test_indices]
)[:, 1]

leaky_auc = roc_auc_score(
    y.iloc[test_indices],
    leaky_probabilities
)

score_comparison = pd.DataFrame(
    {
        "experiment": [
            "Honest five-feature model",
            "Model with label-derived leakage"
        ],
        "test_roc_auc": [
            round(honest_auc, 4),
            round(leaky_auc, 4)
        ]
    }
)

display(score_comparison)

print(
    f"Honest test ROC-AUC: {honest_auc:.4f}"
)

print(
    f"Leaky test ROC-AUC: {leaky_auc:.4f}"
)

print(
    "\nThe leaked score is misleading because label_copy_leak "
    "contains the answer itself and would not be available "
    "at the March 24 decision moment."
)

# ---------------------------------------------------------
# 3. Remove the leaked column and keep the honest frame
# ---------------------------------------------------------

X_leaky = X_leaky.drop(columns=["label_copy_leak"])

assert "label_copy_leak" not in X_leaky.columns
assert list(X_leaky.columns) == FEATURE_COLUMNS

final_honest_auc = honest_auc

print(
    "\nPASS: The label-derived feature was deleted."
)

print(
    f"Final reported score is the honest ROC-AUC: "
    f"{final_honest_auc:.4f}"
)

,experiment,test_roc_auc
0,Honest five-feature model,0.6249
1,Model with label-derived leakage,1.0000


Honest test ROC-AUC: 0.6249
Leaky test ROC-AUC: 1.0000

The leaked score is misleading because label_copy_leak contains the answer itself and would not be available at the March 24 decision moment.

PASS: The label-derived feature was deleted.
Final reported score is the honest ROC-AUC: 0.6249


## 4. Data limits

### Data limitations

The main limitation is that this analysis uses only one development month and a restricted subset of content items.

Only 3,611,061 of 9,841,378 March rows, or 36.69%, had `gsc_data_available IS TRUE`. The final feature frame became smaller because it required all 31 March dates, positive historical impressions, and at least one click in the recent seven-day window. The results therefore do not represent content with incomplete history, missing GSC access, or no recent clicks.

The label is also only a directional proxy. A decline in next-week clicks does not prove that the content needs a refresh, because clicks may change due to seasonality, search demand, ranking changes, or other factors outside the content itself.

The honest ROC-AUC of 0.6280 was measured using a random train-test split within March. The same clients may appear in both sets, so this score should be treated as an initial decision-support result rather than proof that the model will generalize to unseen clients or future months.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Summarize measured limitations using results already created above.
# No additional verification query is performed here.

march_total_rows = int(
    availability_check.loc[0, "total_rows"]
)

march_available_rows = int(
    availability_check.loc[0, "rows_after_gsc_is_true"]
)

availability_percent = float(
    availability_check.loc[0, "percent_surviving_availability"]
)

limitation_summary = pd.DataFrame(
    {
        "measured_limit": [
            "Total March daily rows",
            "Rows with GSC data available",
            "Percent surviving availability filter",
            "Rows in final feature frame",
            "Development months used",
            "Honest test ROC-AUC",
        ],
        "value": [
            f"{march_total_rows:,}",
            f"{march_available_rows:,}",
            f"{availability_percent:.2f}%",
            f"{len(feature_frame):,}",
            "1 (March 2026)",
            f"{final_honest_auc:.4f}",
        ],
    }
)

display(limitation_summary)

print(
    "LIMITATION: Results describe a restricted March 2026 slice "
    "and may not generalize to incomplete-history content, "
    "unseen clients, or future months."
)

,measured_limit,value
0,Total March daily rows,"9,841,378"
1,Rows with GSC data available,"3,611,061"
2,Percent surviving availability filter,36.69%
3,Rows in final feature frame,"27,198"
4,Development months used,1 (March 2026)
5,Honest test ROC-AUC,0.6249


LIMITATION: Results describe a restricted March 2026 slice and may not generalize to incomplete-history content, unseen clients, or future months.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.